# PE_V2X_Reliability 复现运行与技术手册（方案A）

本手册面向评审复现与代码阅读：提供路径无关的复现步骤、run_id 输出契约、改参边界，以及逐文件代码说明（顶层脚本 + modules 全覆盖）。

**Date:** 2026-03-01

## 1 复现运行（VS Code 集成终端示例；路径无关）

建议在 VS Code 中打开仓库根目录（记为 `<ROOT>`），然后使用 *Terminal → New Terminal* 打开集成终端。你可以选择 **Command Prompt / PowerShell** 作为终端类型；下述给出两种常见写法（任选其一）。

### 1.1 创建与激活虚拟环境（Windows）

**写法 A：Command Prompt（推荐）**

```bat
cd <ROOT>\03_sim_A\py
python -m venv <ROOT>\.venv
<ROOT>\.venv\Scripts\activate.bat
pip install -r <ROOT>\06_report\requirements_utf8.txt
```

**写法 B：PowerShell（可选）**

```powershell
cd <ROOT>\03_sim_A\py
python -m venv <ROOT>\.venv
<ROOT>\.venv\Scripts\Activate.ps1
pip install -r <ROOT>\06_report\requirements_utf8.txt
```

### 1.2 Smoke（先验证“产物契约”与运行链路）

```text
python run_pipeline_A.py --run_id A_Smoke --preset RefPlus --scenarios Ref --rets 0,1,2 --seed_start 1 --n_seeds 1 --duration_s 60 --msg_rate_hz 10 --tx_mode mix --tx_k 6 --tx_k_cross 2
```

### 1.3 small / S20

- **small**：将 `--n_seeds` 设为 3–5，用于趋势核查（最重要的是 Fig03/Fig04/05/06/07 的趋势一致性）。  
- **S20**：建议直接以 deliverable 中保存的 `run_commands.txt` 为权威协议快照执行，避免手工漏参或口径不一致。


## 2 run_id 输出契约（验收清单）

每次运行输出目录：`<ROOT>\05_results_A\runs\<run_id>\`

应包含：
- `raw/`：包级 CSV（可能较大）
- `tables/`：聚合 CSV（审阅首选）
- `figures/`：旁证图 F1/F2/F3
- `run_commands.txt`：命令快照（复现证据）

验收建议：
1) 必须存在 `tables/` 与 `run_commands.txt`；
2) UrbMask 场景应生成 `position_heterogeneity__*.csv`；Tunnel 场景应生成 `tunnel_segments__*.csv`；
3) 存储受限时可删除 `raw/`，但建议保留 `tables/` 与 `run_commands.txt`。

## 3 改参边界（建议一次只改一个参数，并做 small-seeds 对照）

- **安全层（推荐）**：只改命令行参数（`rets` / `msg_rate_hz` / `deadline_ms` / `n_seeds` / `enable_congestion`）。
- **中风险**：改 `scenario_*.py` 或传播模块（`prop_city.py` / `prop_tunnel.py`）。
- **高风险**：改 `mac_congestion.py`（会改变 Cong-only 机制结构）；改动后应优先复核 Fig03/Fig04 的趋势一致性。

若需要复现 Fig01–Fig07 的 MATLAB 绘图，可基于 `assets/matlab_cache_raw/` 的 `.mat` 缓存文件在 MATLAB 中重绘；对于评审阅读，PNG 预览与 PDF 定稿已足够。

## 4 逐文件代码说明（顶层脚本 + modules 全覆盖）

下述内容以“定位—关键结构—读写产物—改参入口”为骨架，尽量保证首次阅读即可定位到可修改处与可复现证据。

### 4.1 顶层脚本（03_sim_A/py）

### analyze_metrics_A.py

**定位**：聚合统计：raw→tables（三表体系）并生成审阅友好的 CSV。
- 代码规模：约 **335 行**
- 命令行参数（节选）：`--band_max_m`，`--band_min_m`，`--dist_bin_m`，`--mid_bin_m`，`--min_success_per_bin`，`--min_total_per_bin`，`--retrans`，`--run_id`，`--scenario`，`--u_bin_w`，`--u_max`，`--u_min`

**主要依赖（节选）**：
`__future__.annotations`, `argparse`, `pathlib.Path`, `numpy`, `pandas`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`

**关键结构（用于快速定位）**：
- 函数：
  - `_pick_run_id(arg_run_id)`
  - `_quantiles_ms(x)`
  - `_pick_latest_packets_file(raw_dir, scenario, ret)`
  - `main()`

**可能读写的文件/产物（字符串扫描节选）**：
- `position_heterogeneity__UrbMask__ret{args.retrans}__band{int(band_min)}-{int(band_max)}__{tag}.csv`
- `results_packets__{scenario}__ret{ret}.csv`
- `results_packets__{scenario}__ret{ret}__seed*.csv`
- `summary_metrics__{args.scenario}__ret{args.retrans}__{tag}.csv`
- `tunnel_segments__Tunnel__ret{args.retrans}__band{int(band_min)}-{int(band_max)}__{tag}.csv`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 建议先通过 preset/CLI 控制宏观参数；若修改内部常量，请同步记录到报告与命令快照。

### generate_trajectories_A.py

**定位**：输入生成：生成轨迹 CSV（RefPlus：几何+IDM+信号灯+cross/turn）。
- 代码规模：约 **649 行**
- 命令行参数（节选）：`--cross_enable`，`--cross_half_length_m`，`--dt_s`，`--duration_s`，`--flow_cross_vph`，`--flow_main_vph`，`--i1_x`，`--i2_x`，`--idm_T_s`，`--idm_a_mps2`，`--idm_b_mps2`，`--idm_delta`…

**主要依赖（节选）**：
`__future__.annotations`, `argparse`, `numpy`, `pandas`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`, `run_logging.snapshot_file`, `modules.road_geometry.RefPlusGeometry`, `modules.traffic_signals.SignalPlan`, `modules.traffic_idm.IDMParams`, `modules.traffic_idm.idm_accel`

**关键结构（用于快速定位）**：
- 函数：
  - `_pick_run_id(arg_run_id)`
  - `_ensure_time_key(df)`
  - `_spawn_ok(vlist, min_spawn_gap_m)`
  - `_road_tag_main(direction, lane_id)`
  - `_road_tag_cross(which, direction, lane_id)`
  - `_road_tag_turn(which, turn_kind, lane_id)`
  - `_lane_key(kind, which, direction, lane_id)`
  - `_init_lane_state(geom, n_lanes_per_dir, cross_enable)`
  - `_simulate_refplus_idm(geom, plan_i1, plan_i2, flow_main_vph, flow_cross_vph, p_turn_i1, p_turn_i2, p_right, p_left, veh_length_m…)`
  - `main()`

**可能读写的文件/产物（字符串扫描节选）**：
- `traj__Ref.csv`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 建议先通过 preset/CLI 控制宏观参数；若修改内部常量，请同步记录到报告与命令快照。

### generate_tunnel_config_A.py

**定位**：输入生成：生成 Tunnel 配置 CSV（区间、过渡、退化强度与尾延迟参数）。
- 代码规模：约 **77 行**
- 命令行参数（节选）：`--b_floor`，`--b_peak`，`--bell_gamma`，`--delay_exp_scale_ms`，`--delay_extra_ms`，`--run_id`，`--scenario`，`--severity`，`--transition_m`，`--x0_m`，`--x1_m`

**主要依赖（节选）**：
`__future__.annotations`, `argparse`, `pandas`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`, `run_logging.snapshot_file`, `modules.prop_tunnel.TunnelConfig`

**关键结构（用于快速定位）**：
- 函数：
  - `_pick_run_id(arg_run_id)`
  - `main()`

**可能读写的文件/产物（字符串扫描节选）**：
- `tunnel_config__{args.scenario}.csv`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 建议先通过 preset/CLI 控制宏观参数；若修改内部常量，请同步记录到报告与命令快照。

### generate_urbmask_buildings_A.py

**定位**：输入生成：生成 UrbMask 建筑 CSV（矩形楼块几何 + 高度）。
- 代码规模：约 **83 行**
- 命令行参数（节选）：`--max_h_m`，`--max_height_m`，`--max_w_m`，`--min_h_m`，`--min_height_m`，`--min_w_m`，`--n_blocks`，`--road_length_m`，`--run_id`，`--scenario`，`--seed`，`--x_margin_m`…

**主要依赖（节选）**：
`__future__.annotations`, `argparse`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`, `run_logging.snapshot_file`, `modules.buildings_3d.generate_buildings`, `modules.buildings_3d.save_buildings_csv`

**关键结构（用于快速定位）**：
- 函数：
  - `_pick_run_id(arg_run_id)`
  - `main()`

**可能读写的文件/产物（字符串扫描节选）**：
- `buildings__{args.scenario}__seed{args.seed}.csv`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 建议先通过 preset/CLI 控制宏观参数；若修改内部常量，请同步记录到报告与命令快照。

### paths_A.py

**定位**：路径约定：统一管理工程目录与命名规则。
- 代码规模：约 **156 行**

**主要依赖（节选）**：
`__future__.annotations`, `dataclasses.dataclass`, `pathlib.Path`, `datetime.datetime`, `json`

**关键结构（用于快速定位）**：
- 类：
  - `BasePathsA`
  - `RunPathsA`
- 函数：
  - `_detect_project_root(from_file)`
  - `get_base_paths_a()`
  - `make_run_id(prefix)`
  - `ensure_base_dirs_a()`
  - `ensure_run_dirs_a(run_id, save_as_latest, meta)`
  - `load_latest_run_id()`

**可能读写的文件/产物（字符串扫描节选）**：
- `LATEST_RUN.json`
- `run_manifest.json`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 建议先通过 preset/CLI 控制宏观参数；若修改内部常量，请同步记录到报告与命令快照。

### plot_figures_A.py

**定位**：旁证出图：从 tables 生成 deliverable 图（F1/F2/F3）。
- 代码规模：约 **264 行**
- 命令行参数（节选）：`--cdf_max_dist_m`，`--curve_style`，`--min_bin_count`，`--retrans`，`--run_id`，`--scenario`，`--smooth_overlay_raw`，`--smooth_window_m`，`--x_max_m`

**主要依赖（节选）**：
`__future__.annotations`, `argparse`, `pathlib.Path`, `numpy`, `pandas`, `matplotlib.pyplot`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `run_logging.log_command`, `run_logging.update_manifest`

**关键结构（用于快速定位）**：
- 函数：
  - `_pick_run_id(arg_run_id)`
  - `ecdf(x)`
  - `_pick_latest_summary_file(tables_dir, scenario, ret)`
  - `_pick_latest_packets_file(raw_dir, scenario, ret)`
  - `_smooth_by_distance(x, y, window_m)`
  - `main()`

**可能读写的文件/产物（字符串扫描节选）**：
- `F1_PDR_vs_distance__{scenario}__ret{ret}__{tag}.png`
- `F2_Delay_CDF__{scenario}__ret{ret}__{tag_f2}.png`
- `F3_Delay_p95_p99_vs_distance__{scenario}__ret{ret}__{tag}.png`
- `results_packets__{scenario}__ret{ret}.csv`
- `summary_metrics__{scenario}__ret{ret}__*.csv`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 建议先通过 preset/CLI 控制宏观参数；若修改内部常量，请同步记录到报告与命令快照。

### progress_util.py

**定位**：进度工具：长任务进度条与耗时显示。
- 代码规模：约 **43 行**

**主要依赖（节选）**：
`__future__.annotations`, `typing.Iterable`, `typing.Iterator`, `typing.Optional`, `typing.TypeVar`, `sys`

**关键结构（用于快速定位）**：
- 函数：
  - `progress(it, total, desc)`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 建议先通过 preset/CLI 控制宏观参数；若修改内部常量，请同步记录到报告与命令快照。

### run_logging.py

**定位**：审计记录：写 run_commands / manifest 等。
- 代码规模：约 **97 行**

**主要依赖（节选）**：
`__future__.annotations`, `json`, `os`, `shutil`, `sys`, `datetime.datetime`, `pathlib.Path`, `typing.Any`, `typing.Dict`, `typing.Optional`

**关键结构（用于快速定位）**：
- 函数：
  - `_now()`
  - `log_command(run_id, run_results_dir, extra)`
  - `update_manifest(manifest_path, patch)`
  - `snapshot_file(src, run_results_dir, category, rename_to)`
  - `_quote(s)`

**可能读写的文件/产物（字符串扫描节选）**：
- `run_commands.txt`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 建议先通过 preset/CLI 控制宏观参数；若修改内部常量，请同步记录到报告与命令快照。

### run_pipeline_A.py

**定位**：总入口/编排器：统一 run_id 协议，串联输入生成→仿真→聚合→出图→审计快照。
- 代码规模：约 **533 行**
- 命令行参数（节选）：`--buildings_seed`，`--cross_enable`，`--cross_half_length_m`，`--cs_alpha`，`--cs_beta_delay_ms`，`--cs_cbr_cap`，`--cs_exp_scale_ms`，`--cs_gamma_cbr_col`，`--cs_gamma_cbr_delay`，`--cs_mac_efficiency`，`--cs_min_speed_mps`，`--cs_phy_overhead_us`…

**主要依赖（节选）**：
`__future__.annotations`, `argparse`, `subprocess`, `sys`, `pathlib.Path`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `run_logging.log_command`, `run_logging.update_manifest`

**关键结构（用于快速定位）**：
- 函数：
  - `_run(cmd, cwd)`
  - `main()`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 入口建议：优先通过 `run_pipeline_A.py` 的 CLI 参数改参（`--scenarios/--rets/--n_seeds/--msg_rate_hz/--deadline_ms/--enable_congestion` 等）。

### sim_v2x_A.py

**定位**：仿真核心：生成包级样本（distance/success/delay）并写入遮挡/隧道/拥塞证据字段。
- 代码规模：约 **642 行**
- 命令行参数（节选）：`--attempt_spacing_ms`，`--buildings_path`，`--buildings_seed`，`--cs_alpha`，`--cs_beta_delay_ms`，`--cs_cbr_cap`，`--cs_exp_scale_ms`，`--cs_gamma_cbr_col`，`--cs_gamma_cbr_delay`，`--cs_mac_efficiency`，`--cs_min_speed_mps`，`--cs_phy_overhead_us`…

**主要依赖（节选）**：
`__future__.annotations`, `argparse`, `dataclasses.dataclass`, `pathlib.Path`, `typing.Iterable`, `typing.Optional`, `numpy`, `pandas`, `progress_util.progress`, `paths_A.ensure_run_dirs_a`, `paths_A.make_run_id`, `paths_A.load_latest_run_id`, `paths_A.ensure_base_dirs_a`, `paths_A.get_base_paths_a`, `modules.mac_congestion.CongestionParams`, `modules.mac_congestion.compute_ncs_from_distances`, `modules.mac_congestion.compute_airtime_s`, `modules.mac_congestion.compute_cbr`, `modules.mac_congestion.p_collision_from_ncs`, `modules.mac_congestion.congestion_extra_delay_ms` …

**关键结构（用于快速定位）**：
- 类：
  - `Rect`
- 函数：
  - `clamp01(x)`
  - `load_traj(traj_path)`
  - `load_buildings(buildings_path)`
  - `_legacy_dirs()`
  - `_pick_run_id(arg_run_id)`
  - `parse_tx_ids(s, all_ids)`
  - `_tag_is_cross(tag, prefixes)`
  - `compute_delay_ms(distance_m, attempt_idx, attempt_spacing_ms, rng, impairment_b, extra_ms, exp_scale_ms)`
  - `simulate_one_seed(scenario, retrans, seed, msg_rate_hz, tx_ids_fixed, tx_mode, tx_k, tx_k_cross, tx_cross_prefixes, traj…)`
  - `main()`

**可能读写的文件/产物（字符串扫描节选）**：
- `buildings__UrbMask__seed{seed_use}.csv`
- `results_packets__{args.scenario}__ret{args.retrans}__{seed_tag}.csv`
- `traj__Ref.csv`
- `traj__{args.scenario}.csv`
- `tunnel_config__Tunnel.csv`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 建议先通过 preset/CLI 控制宏观参数；若修改内部常量，请同步记录到报告与命令快照。

### 4.2 模块（03_sim_A/py/modules）

### road_geometry.py

**定位**：道路几何：中心线/车道/路口/支路等几何构建与查询。
- 代码规模：约 **194 行**

**主要依赖（节选）**：
`__future__.annotations`, `dataclasses.dataclass`, `typing.Union`, `numpy`

**关键结构（用于快速定位）**：
- 类：
  - `RefPlusGeometry`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 建议先通过 preset/CLI 控制宏观参数；若修改内部常量，请同步记录到报告与命令快照。

### mac_congestion.py

**定位**：拥塞代理：n_cs/CBR/p_col 与 cong_delay_ms（含尾部结构）。
- 代码规模：约 **166 行**

**主要依赖（节选）**：
`__future__.annotations`, `dataclasses.dataclass`, `typing.Optional`, `numpy`

**关键结构（用于快速定位）**：
- 类：
  - `CongestionParams`
- 函数：
  - `compute_airtime_s(pkt_bytes, phy_rate_mbps, mac_efficiency, phy_overhead_us)`
  - `compute_cbr(n_cs, msg_rate_hz, airtime_s, cbr_cap)`
  - `p_collision_from_ncs(n_cs, alpha_col, cbr, gamma_cbr_col)`
  - `congestion_extra_delay_ms(rng, n_cs, beta_delay_ms, exp_scale_ms, cbr, gamma_cbr_delay)`
  - `compute_ncs_from_distances(dist_all, tx_index, r_cs_m, active_mask, speed_all, min_speed_mps)`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 关注：`n_cs→CBR→p_col` 映射与 `cong_delay_ms` 的均值/尾部；改动后必须复核 Fig03/Fig04。

### buildings_3d.py

**定位**：建筑几何：楼块生成与几何查询（支撑 d_min 计算）。
- 代码规模：约 **163 行**

**主要依赖（节选）**：
`__future__.annotations`, `dataclasses.dataclass`, `pathlib.Path`, `typing.Iterable`, `numpy`, `pandas`

**关键结构（用于快速定位）**：
- 类：
  - `Rect2D`
  - `BuildingBlock`（继承：Rect2D）
- 函数：
  - `_pick_zone_by_x(x_mid, road_length_m)`
  - `generate_buildings()`
  - `buildings_to_dataframe(buildings)`
  - `save_buildings_csv(buildings, path)`
  - `load_buildings_csv(path)`
  - `as_rects(buildings)`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 建议先通过 preset/CLI 控制宏观参数；若修改内部常量，请同步记录到报告与命令快照。

### prop_city.py

**定位**：城市退化：d_min→b→LOS/NLOS 混合，可选反射等效项。
- 代码规模：约 **140 行**

**主要依赖（节选）**：
`__future__.annotations`, `typing.Iterable`, `typing.Protocol`, `typing.Tuple`, `numpy`

**关键结构（用于快速定位）**：
- 类：
  - `RectLike`（继承：Protocol）
- 函数：
  - `clamp01(x)`
  - `segment_intersects_rect(ax, ay, bx, by, rect)`
  - `segment_to_rect_min_distance(ax, ay, bx, by, rect)`
  - `blockage_strength_with_dmin(ax, ay, bx, by, buildings, transition_m)`
  - `p_success_los(distance_m)`
  - `p_success_nlos(distance_m)`
  - `refl_gain_db(d_min_m, b, gmax_db, d0_m)`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 关注：`d_min→b` 的映射尺度（如 `T/urb_transition_m`）、LOS/NLOS 曲线、反射等效项系数。

### prop_tunnel.py

**定位**：隧道退化：tunnel_u 位置；bell+fade 退化；尾延迟注入。
- 代码规模：约 **111 行**

**主要依赖（节选）**：
`__future__.annotations`, `dataclasses.dataclass`, `pathlib.Path`, `typing.Tuple`, `numpy`, `pandas`

**关键结构（用于快速定位）**：
- 类：
  - `TunnelConfig`
- 函数：
  - `clamp01(x)`
  - `tunnel_impairment_b(tx_x, rx_x, cfg)`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 关注：隧道区间、过渡长度、强度参数（`b_floor/b_peak/γ`）与尾延迟尺度。

### scenario_refplus.py

**定位**：场景 preset：冻结默认参数（几何/交通/传播/隧道/拥塞）。
- 代码规模：约 **65 行**

**主要依赖（节选）**：
`__future__.annotations`, `dataclasses.dataclass`, `dataclasses.asdict`, `dataclasses.field`

**关键结构（用于快速定位）**：
- 类：
  - `RefPlusGeometryConfig`
  - `TrafficIDMConfig`
  - `TrafficSignalConfig`
  - `RefPlusScenarioConfig`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 在 `scenario_*.py` 中改默认参数（几何/交通/传播/隧道/拥塞）；改动后优先复核 Fig03/04/05/06/07 的趋势一致性。

### traffic_signals.py

**定位**：交通动力学：IDM 跟驰与信号灯相位（产生车团/排队/释放）。
- 代码规模：约 **54 行**

**主要依赖（节选）**：
`__future__.annotations`, `dataclasses.dataclass`

**关键结构（用于快速定位）**：
- 类：
  - `SignalPlan`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 建议先通过 preset/CLI 控制宏观参数；若修改内部常量，请同步记录到报告与命令快照。

### scenario_urbmaskplus.py

**定位**：场景 preset：冻结默认参数（几何/交通/传播/隧道/拥塞）。
- 代码规模：约 **43 行**

**主要依赖（节选）**：
`__future__.annotations`, `dataclasses.dataclass`, `dataclasses.asdict`, `dataclasses.field`

**关键结构（用于快速定位）**：
- 类：
  - `UrbMaskBuildingsConfig`
  - `UrbMaskPropagationConfig`
  - `UrbMaskScenarioConfig`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 在 `scenario_*.py` 中改默认参数（几何/交通/传播/隧道/拥塞）；改动后优先复核 Fig03/04/05/06/07 的趋势一致性。

### traffic_idm.py

**定位**：交通动力学：IDM 跟驰与信号灯相位（产生车团/排队/释放）。
- 代码规模：约 **38 行**

**主要依赖（节选）**：
`__future__.annotations`, `dataclasses.dataclass`, `numpy`

**关键结构（用于快速定位）**：
- 类：
  - `IDMParams`
- 函数：
  - `idm_accel(v, dv, gap, p)`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 建议先通过 preset/CLI 控制宏观参数；若修改内部常量，请同步记录到报告与命令快照。

### scenario_tunnelplus.py

**定位**：场景 preset：冻结默认参数（几何/交通/传播/隧道/拥塞）。
- 代码规模：约 **17 行**

**主要依赖（节选）**：
`__future__.annotations`, `dataclasses.dataclass`, `dataclasses.asdict`, `dataclasses.field`, `prop_tunnel.TunnelConfig`

**关键结构（用于快速定位）**：
- 类：
  - `TunnelScenarioConfig`

**改参入口（建议一次只改一个参数，并做 small-seeds 对照）**：
- 在 `scenario_*.py` 中改默认参数（几何/交通/传播/隧道/拥塞）；改动后优先复核 Fig03/04/05/06/07 的趋势一致性。